MAKE CONTENTS LINKS
MAKE SURE TO MENTION WHERE PRESET FILES ARE GIVEN

In [2]:
# If the cell returns an error, try executing the cell again.

%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


The output of this notebook should be the engineered features numpy file

For lidar:
no input

explain the two lidar and where to download
Give code for checking list of wfs, for checking dates of national lidar and dates of composites for given area
debate the inclusion of the cockspur personal anecdote

output is lidar dsm and dtm clipped to bounding box

For sentinel:
no input

justify date based on previous findings
prove aerial photography is not viable during personal section

output is fully processed and clipped sentinel data (four bands)

For data prep:
input is dsm dtm and sentinel

output is tiles, core features for each tile, feature stack for each tile
(CONTEMPLATE INCLUDING 

In [28]:
# Standard Libraries
import os
import sys
import json
import requests
import math
import zipfile
from pathlib import Path
import pickle

# Geospatial and Image Processing Libraries
import rasterio
from rasterio.windows import from_bounds
import cv2 
import numpy as np

# Scikit-Image and SciPy
from skimage.feature import graycomatrix, graycoprops
from skimage.util import view_as_windows
from skimage.filters.rank import entropy, minimum
from skimage.morphology import disk
from scipy.ndimage import median_filter, minimum_filter

# Web Services
from owslib.wfs import WebFeatureService
from owslib.wcs import WebCoverageService

In [27]:
# Modules

# Adds the parent directory to the path so it can find utils.py or src/
sys.path.append(os.path.abspath('..'))

import src.utils as utils
import src.feature_generation as feature_generation
import params

The following notebook covers all pre-processing steps required for the Random Forest model. This involves downloading and preparing the input data (LiDAR and Multispectral image data), in addition to feature engineering.

For an overview of the full pipeline, please check the README file.

<span style="color:red">Verify whether sticking to any projection would work, or only EPSG:27700</span>

# 1. Downloading and Processing the Input Data

## 1.1 Introduction to the Data Types
The GLA 2024 methodology (§1.2) introduces three categories of data: aerial imagery, LiDAR data, and vector maps. Since the options used for each are not freely available, other alternatives are explored.

1. Aerial imagery.
   
   This includes colour (RGB) and infrared imagery. The GLA method uses commercial Bluesky International data at 25cm resolution. One alternative is Sentinel-2 satellite imagery. The highest available resolution is 10m. A large part of the project is dedicated to compensating for the resolution difference (10m vs. 25cm).

2. LiDAR height data.

   The GLA method uses Digital Surface Model (DSM) and Digital Terrain Model (DTM) data from Bluesky International at 2m and 5m resolution, respectively. A great alternative is the LiDAR data from the UK DEFRA survey, which has several options available at 1m resolution.

3. Vector mapping data.

   Additional vector data from the Ordnance Survey Master Map (OSMM) is used for enhancing the final output. We will use alternative maps from Ordnance Survey and other providers such as Geofabrik.

(WRITTEN BY AI AS PLACEHOLDER. MUST CHANGE)

The GLA 2024 methodology (§1.2) establishes a framework based on three primary data pillars: multi-spectral aerial imagery, topographic LiDAR, and vector mapping. To democratize this analysis, this project replaces proprietary commercial datasets with Free and Open-Source Software (FOSS) alternatives, necessitating a recalibration of the processing pipeline to account for varying spatial resolutions.

1. Multi-spectral Imagery (Sentinel-2 vs. Bluesky):

    While the official GLA methodology utilizes 25cm commercial imagery, this implementation leverages Sentinel-2 satellite data. Although the resolution is limited to 10m, the inclusion of the Near-Infrared (NIR) band remains critical for Vegetation Index calculation. A core technical objective of this replication is the development of a feature engineering workflow designed to mitigate this 40-fold resolution disparity.

2. Topographic Data (DEFRA LiDAR):

    High-resolution surface modeling is achieved through DEFRA’s LiDAR Composite, substituting the 2m/5m Bluesky DSM and DTM. By utilizing 1m resolution DEFRA data, the pipeline maintains—and in some areas exceeds—the topographic precision of the original report, facilitating accurate Height-Above-Ground (HAG) calculations.

3. Vector Mapping and Boundary Definition:

    In place of Ordnance Survey Master Map (OSMM) products, this analysis incorporates open-access vector data from Ordnance Survey Open Map Local and Geofabrik (OpenStreetMap). These layers provide the necessary spatial masks for land-use attribution and final output enhancement.

<span style="color: red">

To transition into a more focused third-person writing style, a few points must be considered.

1. The writer must transition from referring to what "they" did to what "the project" is.
2. Instead of using generic verbs like 'use' or 'has', the writer can employ descriptive verbs such as 'utilise', 'incorporates', and 'features'.
3. Connect every choice with the technical goal immediately.

</span>

* **Important Note**: For the functions to perform as expected, the processed input data from this section must be stored in the following folder:

In [23]:
print(f"Processed input data storage folder: {params.full_core_features_folder()}")

Processed input data storage folder: ..\data\westminster_data\01_data_acquisition_and_preparation\full_core_features


## 1.2 LiDAR Data

### 1.2.1 Downloading the LiDAR Data

The UK Department for Environment, Food & Rural Affairs (DEFRA) offer high resolution LiDAR data, free to download for both **DTM** and **DSM**. The data Is available for browsing and downloading from their survey page: https://environment.data.gov.uk/survey. The data may be downloaded on a tile-per-tile basis. Each tile is a section of the British National Grid (**BNG**) or Ordnance Survey National Grid. For more information, please visit https://en.wikipedia.org/wiki/Ordnance_Survey_National_Grid.

* **Tip**: If you find that the website does not load on your browser, try using an 'incognito' tab. Alternatively, you can clear your browser cookies.

<center>
  <figure>
    <img src="../images/defra-survey-website.png" alt="The Defra Survey Data Download portal" width="70%">
    <figcaption><i>Source: The Defra Survey Data Download portal.</i></figcaption>
  </figure>
</center>

The website offers options for Aerial Imagery, LiDAR data, among others. The function ``print_defra_lidar_indices`` shows all the surveys offered by DEFRA.



In [18]:
utils.print_defra_lidar_indices()

0.	 dataset-9f0fa3fc-a860-4729-adc9-47fe53f658d0:Bathymetry_multibeam_index_catalogue
1.	 dataset-9f0fa3fc-a860-4729-adc9-47fe53f658d0:CASI_Index_Catalogue
2.	 dataset-9f0fa3fc-a860-4729-adc9-47fe53f658d0:LIDAR_Composite_1m_DTM_2022_extents
3.	 dataset-9f0fa3fc-a860-4729-adc9-47fe53f658d0:LIDAR_Composite_1m_First_Return_DSM_2022_extents
4.	 dataset-9f0fa3fc-a860-4729-adc9-47fe53f658d0:LIDAR_Composite_1m_Last_Return_DSM_2022_extents
5.	 dataset-9f0fa3fc-a860-4729-adc9-47fe53f658d0:LIDAR_Composite_2m_DTM_2022_extents
6.	 dataset-9f0fa3fc-a860-4729-adc9-47fe53f658d0:LIDAR_Composite_2m_First_Return_DSM_2022_extents
7.	 dataset-9f0fa3fc-a860-4729-adc9-47fe53f658d0:LIDAR_Composite_2m_Last_Return_DSM_2022_extents
8.	 dataset-9f0fa3fc-a860-4729-adc9-47fe53f658d0:LIDAR_DSM_Time_Stamped_Extents
9.	 dataset-9f0fa3fc-a860-4729-adc9-47fe53f658d0:LIDAR_DTM_Time_Stamped_Extents
10.	 dataset-9f0fa3fc-a860-4729-adc9-47fe53f658d0:LIDAR_Point_Cloud_Index_Catalogue
11.	 dataset-9f0fa3fc-a860-4729-adc9-47f

There are two options of interest for LiDAR data: LiDAR Composite and National LiDAR Programme. Each has its own technical quirks.

1. **LiDAR Composite Data**

    The LiDAR Composite is derived from merging surveys of multiple years into a single raster. The composite was produced in 2022, but the actual dates of the surveys that compose it differ depending on the area. The function ``view_lidar_composite_dates`` prints all the times a survey was taken on an area, given its spatial extent.

* **Tip**: The function receives the spatial extent using the British National Grid (BNG) system, also known as OSGB36 or the EPSG:27700 projection. The BNG spatial extent of your area of interest may be calculated in any GIS software, as long as the projection is in EPSG:27700. All data in this project is projected to EPSG:27700.


In [7]:
# The BNG spatial extent of the City of Westminster is (523843, 177737, 531169, 183893)

spatial_extent = (523843, 177737, 531169, 183893)  # (minX, minY, maxX, maxY)
utils.view_lidar_composite_dates(bbox=spatial_extent, type='DSM')

1.	Filename: FZ_DSM_P_10752 | Start date: 2018-01-18T00:00:00Z | End date: 2018-01-18T00:00:00Z | Resolution: 1
2.	Filename: FZ_DSM_P_10768 | Start date: 2017-12-07T00:00:00Z | End date: 2018-01-24T00:00:00Z | Resolution: 1
3.	Filename: FZ_DSM_P_12151 | Start date: 2020-12-12T00:00:00Z | End date: 2020-12-12T00:00:00Z | Resolution: 1
4.	Filename: FZ_DSM_P_12150 | Start date: 2020-12-04T00:00:00Z | End date: 2021-02-21T00:00:00Z | Resolution: 1


2. **National LiDAR Programme**

    Unlike the LiDAR Composite, the National LiDAR Programme for any given year is constructed from a single survey. The website offers a range of surveys from 2016 to 2023. Despite this, no single area is guaranteed to have surveys on every date. The function ``view_national_lidar_programme_dates`` prints all the BNG tiles that were surveyed, and the year they were surveyed.

In [29]:
# The BNG spatial extent of the City of Westminster is (523843, 177737, 531169, 183893)

spatial_extent = (523843, 177737, 531169, 183893)  # (minX, minY, maxX, maxY)
utils.view_national_lidar_programme_dates(bbox=spatial_extent)

1.	Tile: TQ2575 | Date: 18-017 [ 2018/01/24 ], 17-279 [ 2017/12/07 ] | Resolution: 1
2.	Tile: TQ2580 | Date: 18-017 [ 2018/01/24 ], 17-279 [ 2017/12/07 ] | Resolution: 1
3.	Tile: TQ3075 | Date: 18-017 [ 2018/01/24 ], 17-279 [ 2017/12/07 ] | Resolution: 1
4.	Tile: TQ3080 | Date: 18-017 [ 2018/01/24 ], 17-279 [ 2017/12/07 ] | Resolution: 1
5.	Tile: TQ2075 | Date: 18-012 [ 2018/01/18 ] | Resolution: 1
6.	Tile: TQ2080 | Date: 18-012 [ 2018/01/18 ] | Resolution: 1
7.	Tile: TQ2075 | Date: 21-037 [ 2021/02/21 ], 20-210 [ 2020/12/04 ] | Resolution: 1
8.	Tile: TQ2080 | Date: 21-037 [ 2021/02/21 ], 20-210 [ 2020/12/04 ] | Resolution: 1
9.	Tile: TQ2575 | Date: 20-212 [ 2020/12/12 ] | Resolution: 1
10.	Tile: TQ2580 | Date: 20-212 [ 2020/12/12 ] | Resolution: 1
11.	Tile: TQ3075 | Date: 20-212 [ 2020/12/12 ] | Resolution: 1
12.	Tile: TQ3080 | Date: 20-212 [ 2020/12/12 ] | Resolution: 1
13.	Tile: TQ2075 | Date: 23-057 [ 2023/03/15 ], 23-003 [ 2023/01/09 ], 23-007 [ 2023/01/14 ], 23-008 [ 2023/01/16 ]

-----

**Westminster Project**

Of the two types of LiDAR data, the National LiDAR Programme data is recommended. Initially, the LiDAR 2022 composite was processed and integrated into the project. It was later found that the composite under-detected the canopy by roughy 50%. This is most likely a processing decision by DEFRA. After facing several classification hurdles, the project switched to using National LiDAR Programme data, which yielded far superior results. The switch was discussed in SECTION x.x.


<center>
  <figure>
    <img src="../images/classification-original-lidar-overlap-hyde-park.png" alt="Canopy classified using 2022 composite" width="40%">
    <img src="../images/classification-original-lidar-2018-overlap-hyde-park.png" alt="2018 National Programme data" width="40%">
    <figcaption><i>Canopy classified using 2022 composite (left) Vs. 2018 National Programme data (right)</i></figcaption>
  </figure>
</center>


As for the type of DSM data, **First Return** was chosen over **Last Return**. First Return represents the highest surface hit by the laser pulse sent by the satellite, returning the "true height" of the object. Last Return, on the other hand, represents the lowest surface the pulse could reach before reflecting, which can underestimate the height of sparse objects. 

----

### 1.2.2 Processing the LiDAR Data

After downloading the required tiles for both the DSM and the DTM, the data can be dropped inside QGIS for initial processing. The following steps are performed inside QGIS on both the DSM and DTM separately:

1. Merge all the downloaded tiles into a single raster.
2. Remove NoData values.
3. Create a vector that spans the spatial extent of the desired area. This is known as the Minimum Bounding Box (MBB). The MBB must then be buffered by a certain length (e.g., 30m).
4. Clip the merged raster to the MBB+Buffer vector.

* **Tip**: The LiDAR data is clipped with a slight buffer, as some algorithms for feature engineering, such as texture computation, require not only information about the pixel itself, but its neighbours as well. If the raster is clipped to the exact borders, then the pixels at the edges of the border will have many NoData neighbours. This compromises the effectiveness of the algorithms.

The output for both the DSM and DTM should look something like this:

<center>
  <figure>
    <img src="../images/westminster-lidar.png" alt="LiDAR DSM data over the City of Westminster" width="70%">
    <figcaption><i>LiDAR DSM data over the City of Westminster</i></figcaption>
  </figure>
</center>

* City of Westminster borough border is sourced from https://data.london.gov.uk/dataset/london-boroughs-e55pw/.

* **Tip**: To get the true height of objects off the ground, known as **nDSM*, the DTM layer is subtracted from the DSM layer. This can be performed using the ``raster calculator`` in QGIS. While nDSM is computed automatically during [feature engineering] (x.x), it is worthwhile to compute the layer externally for use during [labelling](#2.1.2-Labelling-Rasters-using-the-ThRasE-Plugin).

----
**Modifications Made to the 2018 LiDAR Data**

The 2018 National LiDAR programme image covering Hyde Park in the City of Westminster has what appears to be temporary shelters on the Western side of the park. This would compromise the efficacy of the Machine Learning Model. The solution involved merging the 2022 LiDAR composite image with the 2018 image, utilising a mask to cover the sheltered area. The 'fz-dsm-processed.tif' file given is the 2018 image merged with the 2022 image in the masked area.

<center>
  <figure>
    <img src="../images/hyde-park-2018-before.png" alt="Hyde Park before masking" width="40%">
    <img src="../images/hyde-park-2018-after.png" alt="Hyde Park after masking" width="40%">
    <figcaption><i>Hyde Park before (left) and after (right) masking</i></figcaption>
  </figure>
</center>

The mask used to merge the 2018 and 2022 composite LiDAR images is given at data/westminster_data/01_data_acquisition_and_preparation/hyde-park-mask.gpkg.

----


## 1.3 Satellite data (Aerial Imagery)
### 1.3.1 Downloading the Satellite Data

The GLA 2024 methodology utilises commercial data from Bluesky International Ltd. The resolution is high at 25cm per pixel for colour imagery (RBG) and 50cm per pixel for infrared imagery (§1.2). 

Finding freely available data at this resolution is a challenge. The first resource to check would be the DEFRA survey portal, as they offer vertical photography. The function ``view_vertical_photography_dates`` returns all the images taken in the given spatial extent, and their type (RGB/Nightime).

In [11]:
# The BNG spatial extent of the City of Westminster is (523843, 177737, 531169, 183893)

spatial_extent = (523843, 177737, 531169, 183893)  # (minX, minY, maxX, maxY)
utils.view_vertical_photography_dates(bbox=spatial_extent)

1.	Image type: RGB | Date: 2008 | Res: 0.4m
2.	Image type: RGB | Date: 2008 | Res: 0.4m
3.	Image type: RGB | Date: 2008 | Res: 0.4m
4.	Image type: RGB | Date: 2008 | Res: 0.4m
5.	Image type: RGB | Date: 2008 | Res: 0.4m
6.	Image type: RGB | Date: 2008 | Res: 0.4m
7.	Image type: RGB | Date: 2008 | Res: 0.4m
8.	Image type: RGB | Date: 2008 | Res: 0.4m
9.	Image type: RGB | Date: 2008 | Res: 0.4m
10.	Image type: RGB | Date: 2008 | Res: 0.4m
11.	Image type: RGB | Date: 2008 | Res: 0.4m
12.	Image type: RGB | Date: 2008 | Res: 0.4m
13.	Image type: RGB | Date: 2008 | Res: 0.4m
14.	Image type: RGB | Date: 2008 | Res: 0.4m
15.	Image type: RGB | Date: 2008 | Res: 0.4m
16.	Image type: RGB | Date: 2008 | Res: 0.4m
17.	Image type: NightTime | Date: 2012 | Res: 0.2m
18.	Image type: NightTime | Date: 2012 | Res: 0.2m
19.	Image type: NightTime | Date: 2012 | Res: 0.2m
20.	Image type: NightTime | Date: 2012 | Res: 0.2m
21.	Image type: NightTime | Date: 2012 | Res: 0.2m
22.	Image type: NightTime | Date: 

Not only are most images taken in the late 2000s and early 2010s, but they are largely taken at night. The area of interest for the reader could contain modern photography, but due to the uncertain availability and lack of modern imagery of London, other sources are considered.

Satellite data, like Aerial Imagery, offers colour and infrared in the form of 'bands'. The European Space Agency, or **ESA** for short, offers free satellite imagery from the **Sentinel-2 satellite** at 10m spatial resolution. While it is much lower than Bluesky's offerings, it is the best free and consistent alternative found online. Many technical considerations will be taken throughout this process to compensate for the lower resolution.

Another contender is the **Landsat satellite (8-9)**, but its 30m spatial resolution falls behind that of the Sentinel-2 satellite.

Sentinel-2 satellite imagery can be browsed and downloaded from the official **Copernicus Browser** at https://browser.dataspace.copernicus.eu/. 

<center>
  <figure>
    <img src="../images/copernicus-browser-april-2020-data.png" alt=" ESA Copernicus browser (image of April 2020)" width="70%">
    <figcaption><i>Source: ESA Copernicus browser (image of April 2020)</i></figcaption>
  </figure>
</center>

The level 2 atmospherically corrected (**L2A**) version of the imagery should be downloaded, as it has the clearest image of the surface due to filtering out the effects of the sun, atmosphere, and sensor from the image.

Though the Sentinel-2 satellite offers 13 bands, only the subset responsible for capturing the colour and infrared imagery is incorporated into the project, as per GLA methodology section 1.2. The following are the desired Sentinel-2 bands:

1. Red (B04), Green (B03), and Blue (B02) bands for colour imagery
2. Near Infrared (B08) for colour infrared imagery

* **Tip**: Filter for options with low cloud cover before choosing which imagery to download.

----
**Deciding the date to use for Westminster**

As mentioned in [prior](#1.2.2-Processing-the-LiDAR-Data), the 2022 LiDAR Composite was used initially. Over the spatial extent of the City of Westminster, the function ``view_lidar_composite_dates`` shows that the LiDAR surveys are split between early 2018 and late 2020/early 2021. 

Since the raster offered is a composite of several dates, there is no one correct way to temporally align the Sentinel-2 and LiDAR composite data. The choice must be a compromise between the earliest and latest survey dates. Since most of what the GLA methodology utilised was from April (§2.1), a fair choice of the Sentinel-2 imagery would be from April 2020. 23rd of April was chosen for its low cloud cover; a rarity for UK data.

A large part of the project's validity depends on comparing its canopy estimates to the GLA's. The GLA methodology utilised data mostly from April 2022 (§2.1), whereas the project utilises data from April 2020. Analysis of the GLA's canopy cover estimates over the years alleviates concerns over the temporal inconsistency. In 2019, the GLA reported a canopy cover of 21.06$\pm$0.2% and a green cover of 47.6% to 50.7%, derived from 2016 data (§1.1). In 2024, the GLA reported a canopy cover of 19.56% $\pm$  0.3% and a green cover of 51.68% $\pm$  1.11% (§4.1) derived mostly from April 2022 data. 

There is no drastic change in the estimates between 2016 and 2022, especially when considering the potential errors of these estimates. It is therefore not too far-fetched to use data from 2020 to create a comparable estimate.

----


### 1.3.2 Processing the Satellite Bands
The same processing steps will be applied to each band raster individually. The following functions must be performed in QGIS:

(The sequence of these steps will be discussed shortly)

1. <span style="color: blue">Converting the raster pixels from **Digital Numbers (DNs)** to **Surface Reflectance**.</span>

    The range of possible DN values is inconsistent between images and the bands of the same image. Reflectance, on the other hand, tends to be between 0 and 1. Highly reflective surfaces may exceed a value of 1, and non-reflective surfaces may have values below 0.
   
   The formula to compute the Surface Reflectance **SR** (also known as Bottom of Atmosphere Reflectance or **BOA**) for each band **i** is defined as:

    $$SR_i = \frac{DN_i + \text{BOA\_ADD\_OFFSET}_i}{\text{QUANTIFICATION\_VALUE}}$$

   The formula is applied to the raster using the ``Raster Calculator`` inside QGIS. The variables $\text{BOA\_ADD\_OFFSET}$ and $\text{QUANTIFICATION\_VALUE}$ for each band depend on the processing baseline, which can be found in the XML metadata file of the downloaded product.

2. Clearing **NoData** values.

    At the edges of downloaded satellite rasters, there are usually dark pixels that hold no data, referred to as NoData pixels. These pixels store a value of 0 in Sentinel-2 rasters. If the NoData pixels are left in this state, the Surface Reflectance formula will calculate its reflectance as -0.1. To prevent them from interfering with downstream algorithms, a value of -9999 is assigned to all these pixels. While they would usually be clipped already, it pays to be thorough. Steps 1 and 4 are computed simultaneously by applying the formula: ``("Band@1" > 0) * (("Band@1" - 1000) / 10000.0) + (("Band@1" <= 0) * -9999)`` 
    
    Where ``"Band@1"`` is the Sentinel-2 band. The formula assumes $\text{BOA\_ADD\_OFFSET} = 1000$ and $\text{QUANTIFICATION\_VALUE} = 10,000$.


3. **Reprojecting** to the British National Grid (EPSG:27700).
4. **Clipping** to the MBB+Buffer created [here](#1.2.2-Processing-the-LiDAR-Data).

5. **Resample** the spatial resolution to 1m using bilinear interpolation.

   Resampling ensures the satellite imagery matches the LiDAR's resolution. This step is crucial for the Machine Learning Model.

6. **Snap** the output to the LiDAR data's grid.


Computing **Surface Reflectance** is done first, followed by removing NoData values. The steps that follow may be computed simultaneously using the ``Warp (reproject)`` process in QGIS. Computing each step individually can cause unintended side effects. The rationale is given below.

* **Why Surface Reflectance is Computed First**.
  
    Every other step, except for removing NoData values, warps the image in some way. The image becomes fundamentally different, which may compromise the effectiveness of the Reflectance calculation.

DO THE GRID AND INTERPOLATION EXPLANATIONS LATER

### 1.3.3 Colour Composites

The ``Virtual raster`` process in QGIS is used to construct a **Colour Composite** by assigning Satellite bands to the red, green, and blue channels of the composite. Colour Composites are used in downstream tasks, such as labelling (x.x) and feature engineering (x.x). Below are two powerful common composites:

1. **True Colour (RGB) Composite** (Red: B04, Green: B03, Blue: B02).

   <center>
      <figure>
        <img src="../images/rgb-composite.png" alt=" RGB Composite of the City of Westminster southern section" width="40%">
        <figcaption><i>RGB Composite of the City of Westminster southern section</i></figcaption>
      </figure>
    </center>

2. **Colour Infrared (CIR) Composite** (Red: B08, Green: B04, Blue: B03).

   <center>
      <figure>
        <img src="../images/cir-composite.png" alt=" CIR Composite of the City of Westminster southern section" width="40%">
        <figcaption><i>CIR Composite of the City of Westminster southern section</i></figcaption>
      </figure>
    </center>


* For more information on Colour Composites:

    Creation in QGIS: https://www.youtube.com/watch?v=QbevCXrYkpQ.
  
    Common usages of composites: https://gisgeography.com/sentinel-2-bands-combinations/.

### 1.3.4 Normalized Difference Vegetation Index (NDVI) 

**NDVI** is a metric for quantifying the health of vegetation. It is calculated using the **NIR** and **Red** bands using the following formula: 

$$\text{NDVI} = \frac{\text{NIR} - \text{Red}}{\text{NIR} + \text{Red}}$$

It is especially useful for distinguishing vegetation during labelling (x.x).


<center>
  <figure>
    <img src="../images/ndvi-multiply-ndsm.png" alt=" LiDAR DSM layer with NDVI layer in multiply mode" width="50%">
    <figcaption><i>LiDAR DSM with NDVI layer in multiply mode</i></figcaption>
  </figure>
</center>

* For more information on NDVI:
  
    Raster calculation in QGIS: https://www.youtube.com/watch?v=vpxVkYaWMno.
  
    NASA usage and thresholds: https://science.nasa.gov/earth/earth-observatory/measuring-vegetation-ndvi-evi/.

### 1.3.5 Downloading Historic Google Earth Data (Optional)

Not all real-life imagery appears in the Sentinel-2 imagery due to its low resolution, which may confuse labelling. One solution is to use high-resolution satellite imagery such as **Google Earth**.

Although it may seem ideal, there is a major caveat with labelling using this data. If an object, like a single tree, is too small to show in lower resolution data, then labelling said object as "Tree" would only confuse the Machine Learning. For this reason, using Google Earth data is entirely up to the reader. It is best utilized as a labelling companion rather than the source of truth.

There are several steps required for using Historic Google Earth Imagery in QGIS:

1. Historic satellite imagery may be viewed using the **Google Earth Pro** desktop application. 

   <center>
      <figure>
        <img src="../images/google-earth-april-2020.png" alt=" LiDAR DSM layer with NDVI layer in multiply mode" width="70%">
        <figcaption><i>Source: Google Earth Pro Desktop Application Historic Imagery</i></figcaption>
      </figure>
    </center>
   
3. The chosen imagery is **downloaded** from within the application.

   For more information on downloading historic imagery in Google Earth Pro, please watch https://www.youtube.com/watch?v=xSpHvvgWNxQ.

4. The image is downloaded as a JPEG or PNG without geospatial information. The image must be embedded in the correct geographic location using a **georeferencing** tool, such as the one available in QGIS.

   For more information on georeferencing inside QGIS: https://www.youtube.com/watch?v=XV62QEk0Cxg.

   <center>
      <figure>
        <img src="../images/qgis-georeferencer.png" alt=" Georeferencer using QGIS" width="70%">
        <figcaption><i>Georeferencer using QGIS</i></figcaption>
      </figure>
    </center>

   The larger the image, the more points of reference required for precise georeferencing. It is therefore recommended to only georeference areas the reader wishes to label.

# 2. Preparing Data for the Machine Learning Model
## 2.1 Labelling Data for the Machine Learning Model
The first type of labelled data is the input used to train the Machine Learning Model. The GLA 2024 methodology (§2.3.2) details the creation of 16 **tiles**, each  tile 1000x1000 pixels (250x250 meters). This totals 16 million pixels used as input for the model. The tiles represent the different urban and suburban areas of London. 

* **Important Note**: For the functions to perform as expected, the labels must be stored in the following folder:

In [22]:
print(f"Label storage folder: {params.labelled_tiles_folder()}")

Label storage folder: ..\data\westminster_data\01_data_acquisition_and_preparation\labelled_tiles


### 2.1.1 Size and Spread of Data

One way to adapt the GLA's 2024 methodology is to calculate the density of pixels labelled over London and use that as a standard. The standard found can be applied to any area. The calculation will be carried out on the City of Westminster. The reader may apply the same logic to their desired area.

----
**Calculating the Required Area of Labelled Data for the City of Westminster According to the GLA 2024 Methodology**

The GLA 2024 methodology (§2.3.2) labelled a total of 16 million pixels for training and testing purposes. Since the area of Greater London is 157,200 hectares, it follows that the density of pixels per hectare is:
$$\frac{16,000,000\text{ pixels}}{157,200\text{ hectares}} \approx \mathbf 101.8\text{ pixels/ha}.$$

Since the City of Westminster covers 2,147 ha, the total number of labelled pixels to be used is as follows: 
$$2,147\text{ ha} \times 101.8\text{ pixels/ha} = 218,565\text{ pixels}.$$

[Since each pixel is $1m^2$](#1.3.2-Processing-the-Satellite-Bands), and a single hectare is $10,000m^2$, then it follows that a single hectare would contain 10,000 pixels. The area of training data becomes: 

$$218,565\text{ pixels} \div 10,000\text{ pixels/ha} \approx \mathbf 22\text{ hectares}$$

The data is then split into training and testing portions, with a 70% and 30% split, respectively (§2.3.3). The final total is 15.4 ha of training area and 6.6 ha of testing area. A reasonable way to split the total area into several tiles would be to split the 22 ha into 5 tiles, each of area 4.4 hectares (210x210 meters per tile), making it similar to the area of the tiles used in the GLA's methodology.

The total area of the 5 tiles should span the main features found in the area of interest, such as different types of urban landscapes, vegetation, and canopy cover.

----

* **Note on Computational Requirements**:

  The footnote under page 18 of the GLA 2024 methodology (§2.3.6) mentions that, of the 16 million pixels of labelled data, only a 5 million sample was used for the actual training of the model (beyond hyperparameter tuning) due to technical limitations. The question is whether such a compromise is required in the scope of the reader's project.

  ----
    **Calculating the Computational Requirements of Training a Machine Learning Model for the City of Westminster Area**
  
    22 hectares of labelled data represent 218,565 $1m^2$ pixels. If each pixel contained 43 features (as per §2.3.4), with each feature having a size of 4 bytes (float32 data), then the data array being fed to the model would amount to roughly 36MB of memory ($(218,565 * 43 * 4) / (1028 * 1028)$). A process of such a memory requirement is operable on weaker hardware. 

  ----

### 2.1.2 Labelling Rasters with the ThRasE Plugin

The GLA 2024 methodology (§2.3.2) details the labelling of each pixel to one of three classes: Canopy, Green, and Neither.

The first step of the labelling process is to create an empty raster, one for each tile,  using the ``Create constant raster`` processing tool in QGIS. Each raster should match the size, location, resolution, and projection of its respective tile. The rasters are populated with a NoData value of 0. Since each pixel contains a total of 4 possible values, the raster should be saved with the 'Byte' data type. 

After initialising the empty rasters, a plugin called **ThRasE (Thematic Raster Editor)** is used to "paint on" the classification values. The plugin offers valuable functionality for viewing multiple concurrent windows, each showing a different layer. Such a feature is essential for ease of labelling. For more details on ThRasE, please visit the link: https://smbyc.github.io/ThRasE/introduction.html.

<center>
  <figure>
    <img src="../images/thrase-tile-labelling.png" alt=" Labelling a raster with ThRasE using NDVI, CIR, nDSM, and Maps data" width="70%">
    <figcaption><i>Labelling a raster with ThRasE using NDVI, CIR, nDSM, and Maps data</i></figcaption>
  </figure>
</center>

It is important to follow a consistent labelling framework for the algorithms to perform as expected. The following is the framework used throughout the project. Each pixel of the labelled tile can have one of four classes: 0: NoData, 1: Canopy, 2: Green, 3: Neither. All NoData pixels are converted to one of the other pixels by the end of the labelling process.

| Index | Label |
| :--- | :--- |
| 0 | NoData |
| 1 | Canopy |
| 2 | Green |
| 3 | Neither |


* **Important Note**: Every process that follows will assume the labelled tiles, and the Machine Learning outputs that follow, contain four classes rather than three, due to the existence of NoData values during labelling. Therefore, the 'Canopy' class is indexed at 1 rather than 0. 'Green' has an index of 2, and 'Neither' has an index of 3. Every function is coded with these indices in mind.

<span style="color: red">#### Looping back after performing classification<span>

| Code | Description |
| :--- | :--- |
| 0 | Tree |
| 1 | Vegetation (not tree) |
| 2 | Bare ground |
| 3 | Vegetation (unsure if tree) |
| 4 | Manmade |
| 5 | Unsure |
| 6 | Water |

### 2.1.3 Confirming the Integrity of the Labelled Rasters

Since labelling is a time-consuming process, it pays to be prudent. The following are some guidelines to keep in mind during labelling:
1. The tiles should be on the same grid as the rest of the data (e.g., 1 meter resolution, EPSG:27700 Projection).
2. The file size can be kept small by assigning a 'Byte' data type.
3. All NoData pixels should be labelled before completion.
4. Ensure that the labels are correctly indexed for this project (1: Canopy, 2: Green, 3: Neither).

With this, the labelled rasters should be compatible with the Feature Engineering and Machine Learning processes.

<center>
  <figure>
    <img src="../images/labelled-tiles-over-ndsm.png" alt=" Labelled rasters (tiles) over nDSM layer of the City of Westminster" width="70%">
    <figcaption><i>Labelled rasters (tiles) over nDSM layer of the City of Westminster</i></figcaption>
  </figure>
</center>


## 2.2 Feature Engineering
### 2.2.1 Defining the "Core Features"

[The previous section](#1.-Downloading-and-Processing-the-Input-Data) defined a **set of core features** that are input to the Machine learning (ML) Model:
1. The Red, Green, Blue, and NIR channels from the Sentinel-2 Satellite (or equivalent).
2. The DSM and DTM layers from the DEFRA LiDAR Satellite (or equivalent).

In addition, [a set of labelled rasters](#2.1-Labelling-Data-for-the-Machine-Learning-Model), or 'tiles', were defined as the **ground truth** during ML model training.

Since the core features processed thus far have geographic coordinates embedded in each pixel, the feature generation can be performed using only parts of the core features that overlap the labels. By clipping the core features to the labelled raster, the processing time is drastically reduced, and the features are generated exclusively for the training areas. The pixels of unlabelled areas will have their features generated at [prediction time](x.x), and their features are not stored. 

* **Note**: When clipping the core features to the extent of the tiles, a buffer should be applied to ensure that all pixels on the edges of tiles do not neighbour empty pixels. Feature 26 in Appendix 2 requires 27 pixels in all directions, the most of all the features mentioned. Therefore, the clipping scripts create a 30-pixel buffer around each tile. The buffer is later removed after feature engineering.

The core features are clipped and buffered using ``feature_generation.generate_features()``.

In [29]:
# --- 1. Defining folder paths and file names ---

# Define the folder paths used in section 2.2. Set params.USING_WESTMINSTER_DATA to 'False' if utilising user data. 

labelled_tiles_folder = params.labelled_tiles_folder()
tiled_core_featurs_folder = params.tiled_core_features_folder()
full_core_featurs_folder = params.full_core_features_folder()
feature_stacks_folder = params.feature_stacks_folder()

In [39]:
# Print the folder paths

print(f"The Relevant Input Folders of the Notebook: \n\n"
      f"Labelled tiles:\t\t{labelled_tiles_folder}\n"
      f"Tiled core features:\t{tiled_core_featurs_folder}\n"
      f"Full core features:\t{full_core_featurs_folder}\n"
      f"Feature stacks:\t\t{feature_stacks_folder}")

The Relevant Input Folders of the Notebook: 

Labelled tiles:		..\data\westminster_data\01_data_acquisition_and_preparation\labelled_tiles
Tiled core features:	..\data\westminster_data\01_data_acquisition_and_preparation\tiled_core_features
Full core features:	..\data\westminster_data\01_data_acquisition_and_preparation\full_core_features
Feature stacks:		..\data\westminster_data\01_data_acquisition_and_preparation\feature_stacks


In [31]:
# Extract the tile names from labelled_tiles_folder
tile_names = [f.name for f in labelled_tiles_folder.iterdir() if f.is_file()]

In [32]:
tile_names

['tile-1.tif', 'tile-2.tif', 'tile-3.tif', 'tile-4.tif', 'tile-5.tif']

In [33]:
# Define your core features

# WARNING: The rasters must be in the following order:
# full_core_feature_paths = [
#     'PROCESSED DTM FILE',
#     'PROCESSED DSM FILE',
#     'PROCESSED BLUE BAND FILE',
#     'PROCESSED GREEN BAND FILE',
#     'PROCESSED RED BAND FILE',
#     'PROCESSED NIR BAND FILE'
#     ]

core_feature_names = [
    'dtm-processed.tif',
    'fz-dsm-processed.tif',
    'B02-processed.tif',
    'B03-processed.tif',
    'B04-processed.tif',
    'B08-processed.tif'
    ]

In [36]:
# --- 2. Clip the core features ---

# The clipped core features are stored in 01_data_acquisition_and_preparation\tiled_core_features
feature_generation.clip_rasters_by_tiles(core_features=[Path(full_core_featurs_folder, feature) for feature in core_feature_names],
                                        tiles=[Path(labelled_tiles_folder, tile) for tile in tile_names],
                                        outputs=[Path(params.tiled_core_features_folder(is_output=True), tile).with_suffix('') for tile in tile_names],
                                        buffer_px=30
                                        )

Clipping core features to tile-1.
Core features saved to ..\data\user_data\01_data_acquisition_and_preparation\tiled_core_features\tile-1.

Clipping core features to tile-2.
Core features saved to ..\data\user_data\01_data_acquisition_and_preparation\tiled_core_features\tile-2.

Clipping core features to tile-3.
Core features saved to ..\data\user_data\01_data_acquisition_and_preparation\tiled_core_features\tile-3.

Clipping core features to tile-4.
Core features saved to ..\data\user_data\01_data_acquisition_and_preparation\tiled_core_features\tile-4.

Clipping core features to tile-5.
Core features saved to ..\data\user_data\01_data_acquisition_and_preparation\tiled_core_features\tile-5.



### 2.2.2 Performing Feature Generation

The clipped core features are input to ``feature_generation.generate_features()``, which generates the features mentioned in GLA methodology Appendix 2. The methodology (§2.3.4) states that the features are chosen to closely match those used by Curio in the 2018 analysis, with the addition of new height data not previously employed.

With the exception of features 4 and 5 of Appendix 2, all 43 features are generated. The features, including the labelled tile, are stacked into a single **3D array**, which is the ideal input for the ML model.

* **Note on Excluded Features**:

    The exclusion of features 4 (CIR Composite Blue Band) and 5 (CIR Composite Green Band) from the GLA's methodology is due to the differing nature of the input data. Unlike the original report, where the infrared image was a different data source, Sentinel-2 is the sole provider of aerial imagery. Therefore, in this project, feature 4 is identical to feature 1 (Blue Band), and feature 5 is identical to feature 2 (Green Band).

* **Note on Grey Level Co-Occurrence Matrix**:

    The Grey Level Co-Occurrence Matrix (GLCM) features (34-40) require the most computation by far. Reducing the number of grey levels per pixel from 256 to 32 improves processing speed x64, with very little sacrifice in output quality (especially for 1m resolution). A dedicated helper function ``feature_generation.calculate_glcm_map()`` performs the quantisation alongside GLCM functions. The function is implemented internally within ``feature_generation.generate_features()``.

The GLA methodology (§1.3 & §4.22) states that the model was trained to recognise vegetation as trees if the height exceeds 4 meters. This is considered valuable so as not to classify shrubs and young trees as 'Canopy'. Therefore, to further improve classification, a new feature not mentioned in Appendix 2 is generated. This feature is a binary flag for all pixels whose height is above 4m. To keep with the report's numbering scheme, this feature is numbered 44. The total number of features generated amounts to 42.

``feature_generation.generate_features()`` constructs a 3D Numpy array. After appending the labelled raster as the last layer of the **feature stack**, the shape of the stack becomes (43, 210, 210). In other words, 43 stacked tiles, each serving a function. Using the feature stack, the ML model is given 42 points of reference for classifying a pixel, and 1 ground truth.


<center>
  <figure>
    <img src="../images/multi-band-raster.png" alt=" Multi-band raster. Source: pro.arcgis.com" width="50%">
    <figcaption><i>Multi-band raster. Source: pro.arcgis.com/</i></figcaption>
  </figure>
</center>

In [38]:
# --- 3. Generate features using the clipped rasters ---

# The generated features are stored in 01_data_acquisition_and_preparation\feature_stacks
feature_generation.generate_features(tiled_core_features=tiled_core_featurs_folder,
                                    core_features = core_feature_names,
                                    tiles=[Path(labelled_tiles_folder, tile) for tile in tile_names],
                                    outputs=[Path(params.feature_stacks_folder(is_output=True), f'{Path(tile).stem}_fstack.npy') for tile in tile_names],
                                    buffer_px=30 # Must be the same as clip_rasters_by_tiles()
                                    )

Generating feature stack for tile-1.
Feature stack saved to ..\data\user_data\01_data_acquisition_and_preparation\feature_stacks\tile-1_fstack.npy
Generating feature stack for tile-2.
Feature stack saved to ..\data\user_data\01_data_acquisition_and_preparation\feature_stacks\tile-2_fstack.npy
Generating feature stack for tile-3.
Feature stack saved to ..\data\user_data\01_data_acquisition_and_preparation\feature_stacks\tile-3_fstack.npy
Generating feature stack for tile-4.
Feature stack saved to ..\data\user_data\01_data_acquisition_and_preparation\feature_stacks\tile-4_fstack.npy
Generating feature stack for tile-5.
Feature stack saved to ..\data\user_data\01_data_acquisition_and_preparation\feature_stacks\tile-5_fstack.npy


The feature stack is the sole input to the Machine Learning model in 02_model_training_and_postprocessing.ipynb.

Optionally, ``utils.rasterize_feature_stack()`` may be used to convert a feature stack into a multi-band raster for viewing in any GIS software.

In [40]:
# --- 4. Rasterize a feature stack ---

# Define the name of the feature stack to rasterize. 
name_of_fstack = 'tile-1_fstack'
path = Path(params.feature_stacks_folder(is_output=True), name_of_fstack)

# The rasterized feature stack is stored in 01_data_acquisition_and_preparation\feature_stacks.
utils.rasterize_feature_stack(feature_stack_path=path.with_suffix('.npy'), 
                              metadata_path=path.with_suffix('.pkl'), 
                              output_path=path.with_suffix('.tif'), 
                              buffer_px=30)

Rasterized stack (43 bands) saved: ..\data\user_data\01_data_acquisition_and_preparation\feature_stacks\tile-1_fstack.tif
